# TerraPulse - Landslide Model v0.1

Runs the trained landslide susceptibility model and shows what each alert level
costs in missed landslides versus false alarms.

## Read this before quoting any number

- The score is a **ranking, not a calibrated probability**. The model was trained
  on a 1:1 balanced sample (8,546 recorded landslides against 8,546 generated
  negatives), so the base rate is 0.5 - not the real-world event rate.
- **Accuracy is deliberately not reported.** On a 50/50 sample it is meaningless,
  and on real data, where landslides are rare, a model that warns about nothing
  would score above 99%.
- Performance is measured on points the model **never saw**, using spatial block
  cross-validation. Plain random splits are not used: neighbouring points sit on
  the same slope and would leak across the split.
- The negatives are **constructed, not observed**. GSI is a presence-only
  inventory - nobody records where a landslide did *not* happen.

## 1. Load the model artifact

In [ ]:
import json, os
import numpy as np
import pandas as pd
import joblib

REPO = r"C:\Users\Ravichandran\Downloads\TERRAPULSE"
ART = os.path.join(REPO, "models", "landslide_v0.4")

schema = json.load(open(os.path.join(ART, "feature_schema.json"), encoding="utf-8"))
thresholds = json.load(open(os.path.join(ART, "thresholds.json"), encoding="utf-8"))
model = joblib.load(os.path.join(ART, schema["model"] + ".joblib"))

features = schema["numeric_features"] + schema["categorical_features"]

print("model     :", schema["model"], schema["artifact_version"])
print("features  :", len(features))
print("alert cut :", round(thresholds["f1_optimal_mean_over_folds"], 2))

## 2. Score every point

The derived columns (aspect_sin, aspect_cos, miss_rain_clim, miss_soil) are
built at training time and are not stored in the CSV, so the training module is
imported here to apply exactly the same derivation. If the two ever drifted
apart, the model would quietly be fed different inputs than it was fitted on.

**This cell scores rows the model was already fitted on, so its agreement
figures are optimistic. Section 3 is the honest measurement.**

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "tlm", os.path.join(REPO, "scripts", "train_landslide_model.py"))
tlm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tlm)

data = os.path.join(REPO, "TERRAPULSE_DATA", "13_MODEL_DATASETS",
                    "landslide_dataset_v0.1.csv")
df = tlm.derive_features(pd.read_csv(data))

cut = thresholds["f1_optimal_mean_over_folds"]
df["score"] = model.predict_proba(df[features])[:, 1]
df["warned"] = (df["score"] >= cut).astype(int)

print("points scored :", len(df))
print("warned        :", int(df["warned"].sum()))
print("quiet         :", int((df["warned"] == 0).sum()))

In [ ]:
cols = ["SLIDE_NO", "STATE", "score", "landslide", "warned"]
print("--- highest scores ---")
print(df.sort_values("score", ascending=False).head(8)[cols].to_string(index=False))
print()
print("--- lowest scores ---")
print(df.sort_values("score").head(8)[cols].to_string(index=False))

## 3. The honest measurement - the alert-level dial

Each point below carries the score from a model that **never saw its block**.
That file is written by `scripts/model_operating_points.py`.

The model alone does not decide to warn. It produces a score, and an operator
chooses the cut. That choice trades missed landslides against false alarms.

In [ ]:
oof = pd.read_csv(os.path.join(ART, "oof_predictions.csv"))

cuts = [0.10, 0.20, 0.30, 0.40, 0.50, 0.53, 0.60, 0.70, 0.80, 0.90]
rows = []
for c in cuts:
    w = oof["score"] >= c
    real = oof["landslide"] == 1
    tp = int((real & w).sum())
    fp = int((~real & w).sum())
    fn = int((real & ~w).sum())
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    pre = tp / (tp + fp) if (tp + fp) else 0.0
    f1 = 2 * pre * rec / (pre + rec) if (pre + rec) else 0.0
    rows.append({"cut": c, "warned": int(w.sum()), "caught": tp, "missed": fn,
                 "recall": round(rec, 3), "precision": round(pre, 3),
                 "f1": round(f1, 3)})

dial = pd.DataFrame(rows)
dial

## 4. Choosing a cut in plain terms

- **recall** - of all real landslides, the share we warned about
- **precision** - of all our warnings, the share that were real
- **missed** - real landslides that produced no warning at all

A missed landslide means no warning reached anyone. A false alarm means a team
drives out and finds nothing. Those two errors do not cost the same, which is
why the cut is an operations decision, not a modelling one.

In [ ]:
for label, c in [("cautious - few false alarms", 0.70),
                 ("balanced - current setting", 0.53),
                 ("safety-first - catch almost everything", 0.20)]:
    m = dial[dial["cut"] == c].iloc[0]
    print("%-40s cut %.2f -> catch %3.0f%%, %d false alarms out of %d warnings"
          % (label, c, 100 * m["recall"], m["warned"] - m["caught"], m["warned"]))

## 5. What this model is not

- **Not a calibrated probability.** The 1:1 design fixes the base rate at 0.5.
  The score orders locations by relative susceptibility; it does not say "there
  is a 73% chance this slope fails".
- **Not evidence that a place without a record is safe.** Absence of a GSI
  record is absence of a *survey*, not absence of a landslide.
- **Not a rainfall trigger.** Event-year rainfall was excluded because the
  negatives have no event date, so including it would have leaked the label.
  The model knows *where* landslides happen, not *when*.

## 6. Known weaknesses carried from the training report

- Recall at the balanced cut is about 0.55 - it misses roughly 45% of recorded
  landslides. Not yet good enough to stand alone as an early-warning system.
- Mizoram scored about 0.52 ROC-AUC (a coin flip) when the state was held out
  entirely. That is a generalisation failure, not noise, and it covers 24% of
  the data.
- 67% of GSI records sit within 200 m of a road - partly real (cut slopes),
  partly survey bias, because field teams reach landslides by road. Removing
  all road information still leaves ROC-AUC 0.700, so the model is not purely
  measuring accessibility, but the effect is not zero.

## 7. Running this outside Jupyter

From a terminal, in the repository root:

    python scripts/train_landslide_model.py     # retrain, about 2 min
    python scripts/model_operating_points.py    # the dial table above
    python scripts/predict_landslide.py         # score points

`predict_landslide.py` also takes a file:

    python scripts/predict_landslide.py my_points.csv my_scores.csv

The input must already carry the feature columns listed in
`models/landslide_v0.4/feature_schema.json`. The model does not accept raw
coordinates - a new location needs terrain, rainfall, land cover, soil and
distance features extracted first, which is what `scripts/extract_*.py` do.